In [1]:
import sys
sys.path.append('/home/jovyan/src')

from spark_utils import crear_spark_session
from gold.dimensions import construir_todas_las_dimensiones
from gold.descriptivo import construir_todo_descriptivo

print("Imports OK")

Imports OK


In [1]:
!pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 3.7 MB/s eta 0:00:0000:0100:01


In [4]:
import sys
if '/home/jovyan/src' not in sys.path:
    sys.path.append('/home/jovyan/src')

from spark_utils import crear_spark_session
from gold.descriptivo import construir_viajes_por_hora

spark = crear_spark_session()
construir_viajes_por_hora(spark)
spark.stop()
print("listo!")

2026-06-30 23:00:04,475 [INFO] SparkSession creada: 3.5.0
2026-06-30 23:00:09,521 [INFO] Gold escrito (overwrite): descriptivo/viajes_por_hora (8,762 filas)
2026-06-30 23:00:09,523 [INFO]   viajes_por_hora: yellow 2023 listo
2026-06-30 23:00:10,183 [INFO] Gold escrito (append): descriptivo/viajes_por_hora (8,759 filas)
2026-06-30 23:00:10,184 [INFO]   viajes_por_hora: green 2023 listo
2026-06-30 23:00:11,327 [INFO] Gold escrito (append): descriptivo/viajes_por_hora (8,759 filas)
2026-06-30 23:00:11,329 [INFO]   viajes_por_hora: fhv 2023 listo
2026-06-30 23:00:30,012 [INFO] Gold escrito (append): descriptivo/viajes_por_hora (8,760 filas)
2026-06-30 23:00:30,017 [INFO]   viajes_por_hora: fhvhv 2023 listo
2026-06-30 23:00:34,673 [INFO] Gold escrito (append): descriptivo/viajes_por_hora (8,784 filas)
2026-06-30 23:00:34,674 [INFO]   viajes_por_hora: yellow 2024 listo
2026-06-30 23:00:35,308 [INFO] Gold escrito (append): descriptivo/viajes_por_hora (8,786 filas)
2026-06-30 23:00:35,309 [INF

listo!


In [2]:
import duckdb
import os

con = duckdb.connect()
os.makedirs("/home/jovyan/data/gold/descriptivo/viajes_por_zona", exist_ok=True)

con.execute("""
    COPY (
        SELECT
            COALESCE(o.location_id, d.location_id) as location_id,
            COALESCE(o.anio, d.anio) as anio,
            COALESCE(o.mes, d.mes) as mes,
            COALESCE(o.dia, d.dia) as dia,
            COALESCE(o.tipo_vehiculo, d.tipo_vehiculo) as tipo_vehiculo,
            COALESCE(o.total_viajes_origen, 0) as total_viajes_origen,
            COALESCE(d.total_viajes_destino, 0) as total_viajes_destino
        FROM (
            SELECT
                pu_location_id as location_id,
                anio, mes,
                DAY(pickup_datetime) as dia,
                tipo_vehiculo,
                COUNT(*) as total_viajes_origen
            FROM read_parquet('/home/jovyan/data/silver/trips/*/*/*/*.parquet')
            GROUP BY pu_location_id, anio, mes, DAY(pickup_datetime), tipo_vehiculo
        ) o
        FULL OUTER JOIN (
            SELECT
                do_location_id as location_id,
                anio, mes,
                DAY(pickup_datetime) as dia,
                tipo_vehiculo,
                COUNT(*) as total_viajes_destino
            FROM read_parquet('/home/jovyan/data/silver/trips/*/*/*/*.parquet')
            GROUP BY do_location_id, anio, mes, DAY(pickup_datetime), tipo_vehiculo
        ) d
        ON o.location_id = d.location_id
        AND o.anio = d.anio
        AND o.mes = d.mes
        AND o.dia = d.dia
        AND o.tipo_vehiculo = d.tipo_vehiculo
    ) TO '/home/jovyan/data/gold/descriptivo/viajes_por_zona/'
    (FORMAT PARQUET, PARTITION_BY (anio))
""")

print("viajes_por_zona escrito con DuckDB!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

viajes_por_zona escrito con DuckDB!


In [3]:
import sys
if '/home/jovyan/src' not in sys.path:
    sys.path.append('/home/jovyan/src')

from spark_utils import crear_spark_session
from gold.descriptivo import construir_tendencia_mensual

spark = crear_spark_session()
construir_tendencia_mensual(spark)
spark.stop()
print("tendencia_mensual completo!")

2026-06-30 22:57:11,847 [INFO] SparkSession creada: 3.5.0
2026-06-30 22:57:26,716 [INFO] Gold escrito (overwrite): descriptivo/tendencia_mensual (13 filas)
2026-06-30 22:57:26,753 [INFO]   tendencia_mensual: yellow 2023 listo
2026-06-30 22:57:28,076 [INFO] Gold escrito (append): descriptivo/tendencia_mensual (12 filas)
2026-06-30 22:57:28,078 [INFO]   tendencia_mensual: green 2023 listo
2026-06-30 22:57:30,031 [INFO] Gold escrito (append): descriptivo/tendencia_mensual (12 filas)
2026-06-30 22:57:30,035 [INFO]   tendencia_mensual: fhv 2023 listo
2026-06-30 22:57:53,875 [INFO] Gold escrito (append): descriptivo/tendencia_mensual (12 filas)
2026-06-30 22:57:53,876 [INFO]   tendencia_mensual: fhvhv 2023 listo
2026-06-30 22:58:01,081 [INFO] Gold escrito (append): descriptivo/tendencia_mensual (13 filas)
2026-06-30 22:58:01,083 [INFO]   tendencia_mensual: yellow 2024 listo
2026-06-30 22:58:01,959 [INFO] Gold escrito (append): descriptivo/tendencia_mensual (14 filas)
2026-06-30 22:58:01,963 

tendencia_mensual completo!


In [1]:
import sys
if '/home/jovyan/src' not in sys.path:
    sys.path.append('/home/jovyan/src')

from spark_utils import crear_spark_session
from gold.diagnostico import construir_tarifas_por_zona

spark = crear_spark_session()

construir_tarifas_por_zona(spark)

spark.stop()
print("¡Proceso finalizado!")

2026-07-03 18:46:52,862 [INFO] SparkSession creada: 3.5.0
2026-07-03 18:47:10,857 [INFO] Gold escrito (overwrite): diagnostico/tarifas_por_zona (3,058 filas)
2026-07-03 18:47:10,915 [INFO]   tarifas_por_zona: yellow 2023 listo
2026-07-03 18:47:12,145 [INFO] Gold escrito (append): diagnostico/tarifas_por_zona (2,341 filas)
2026-07-03 18:47:12,146 [INFO]   tarifas_por_zona: green 2023 listo
2026-07-03 18:47:14,139 [INFO] Gold escrito (append): diagnostico/tarifas_por_zona (3,129 filas)
2026-07-03 18:47:14,142 [INFO]   tarifas_por_zona: fhv 2023 listo
2026-07-03 18:47:41,183 [INFO] Gold escrito (append): diagnostico/tarifas_por_zona (3,139 filas)
2026-07-03 18:47:41,194 [INFO]   tarifas_por_zona: fhvhv 2023 listo
2026-07-03 18:47:47,870 [INFO] Gold escrito (append): diagnostico/tarifas_por_zona (3,089 filas)
2026-07-03 18:47:47,873 [INFO]   tarifas_por_zona: yellow 2024 listo
2026-07-03 18:47:48,834 [INFO] Gold escrito (append): diagnostico/tarifas_por_zona (2,474 filas)
2026-07-03 18:47:

¡Proceso finalizado!


In [1]:
import sys
if '/home/jovyan/src' not in sys.path:
    sys.path.append('/home/jovyan/src')

from spark_utils import crear_spark_session
from gold.diagnostico import construir_comparacion_vehiculos

spark = crear_spark_session()

construir_comparacion_vehiculos(spark)

spark.stop()
print("¡Proceso finalizado!")

2026-07-03 19:47:26,345 [INFO] SparkSession creada: 3.5.0
2026-07-03 19:47:44,703 [INFO] Gold escrito (overwrite): diagnostico/comparacion_vehiculos (12 filas)
2026-07-03 19:47:44,727 [INFO]   comparacion_vehiculos: yellow 2023 listo
2026-07-03 19:47:45,678 [INFO] Gold escrito (append): diagnostico/comparacion_vehiculos (12 filas)
2026-07-03 19:47:45,680 [INFO]   comparacion_vehiculos: green 2023 listo
2026-07-03 19:47:47,557 [INFO] Gold escrito (append): diagnostico/comparacion_vehiculos (12 filas)
2026-07-03 19:47:47,558 [INFO]   comparacion_vehiculos: fhv 2023 listo
2026-07-03 19:48:16,459 [INFO] Gold escrito (append): diagnostico/comparacion_vehiculos (12 filas)
2026-07-03 19:48:16,461 [INFO]   comparacion_vehiculos: fhvhv 2023 listo
2026-07-03 19:48:23,052 [INFO] Gold escrito (append): diagnostico/comparacion_vehiculos (12 filas)
2026-07-03 19:48:23,054 [INFO]   comparacion_vehiculos: yellow 2024 listo
2026-07-03 19:48:23,807 [INFO] Gold escrito (append): diagnostico/comparacion_v

¡Proceso finalizado!


In [3]:
import sys
if '/home/jovyan/src' not in sys.path:
    sys.path.append('/home/jovyan/src')

from spark_utils import crear_spark_session
from gold.diagnostico import construir_duracion_distancia

spark = crear_spark_session()

construir_duracion_distancia(spark)

spark.stop()
print("¡Proceso finalizado!")

2026-07-03 19:21:06,636 [INFO] SparkSession creada: 3.5.0
2026-07-03 19:21:13,762 [INFO] Gold escrito (overwrite): diagnostico/duracion_distancia (288 filas)
2026-07-03 19:21:13,764 [INFO]   duracion_distancia: yellow 2023 listo
2026-07-03 19:21:14,371 [INFO] Gold escrito (append): diagnostico/duracion_distancia (288 filas)
2026-07-03 19:21:14,372 [INFO]   duracion_distancia: green 2023 listo
2026-07-03 19:21:15,613 [INFO] Gold escrito (append): diagnostico/duracion_distancia (288 filas)
2026-07-03 19:21:15,615 [INFO]   duracion_distancia: fhv 2023 listo
2026-07-03 19:21:38,880 [INFO] Gold escrito (append): diagnostico/duracion_distancia (288 filas)
2026-07-03 19:21:38,882 [INFO]   duracion_distancia: fhvhv 2023 listo
2026-07-03 19:21:43,269 [INFO] Gold escrito (append): diagnostico/duracion_distancia (288 filas)
2026-07-03 19:21:43,271 [INFO]   duracion_distancia: yellow 2024 listo
2026-07-03 19:21:43,821 [INFO] Gold escrito (append): diagnostico/duracion_distancia (288 filas)
2026-07

¡Proceso finalizado!
